# 4.4: Notebook-Using ColumnTransformer for mixed data

Real-world datasets rarely have just one type of feature. Most contain a mix of numeric columns (that need scaling) and categorical columns (that need encoding). ColumnTransformer is the tool that handles this elegantly.

## Learning outcomes:
- Explain when and why to use ColumnTransformer.
- Apply different preprocessing to numeric and categorical columns.
- Apply ColumnTransformer to a real dataset.
- Use the `remainder` argument to control unprocessed columns (Advanced).
- Build complete pipelines with mixed data types.

Let's start with a simple example, then build up to a complete real-world pipeline.


In [5]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer


## The problem: mixed data types

Let's start with a simple dataset that has both numeric and categorical features:


In [ ]:
# Create a sample dataset with mixed types
df = pd.DataFrame({
    'age': [25, 30, 35, 40, 45],
    'income': [30000, 50000, 75000, 85000, 120000],
    'city': ['NYC', 'LA', 'Chicago', 'NYC', 'LA'],
    'education': ['Bachelor', 'Master', 'Bachelor', 'PhD', 'Master']
})

print("Sample dataset:")
print(df)
print("\nData types:")
print(df.dtypes)


Sample dataset:
   age  income     city education
0   25   30000      NYC  Bachelor
1   30   50000       LA    Master
2   35   75000  Chicago  Bachelor
3   40   85000      NYC       PhD
4   45  120000       LA    Master

Data types:
age           int64
income        int64
city         object
education    object
dtype: object


**The challenge:**
- `age` and `income` are numeric → need scaling.
- `city` and `education` are categorical → need encoding.

We can't apply StandardScaler to categorical columns (it expects numbers). We normally should not apply OneHotEncoder to continuous numeric columns, because it would treat each number as a separate category.

**Solution:** Use ColumnTransformer to apply different preprocessing to different columns.


## Basic ColumnTransformer usage

ColumnTransformer takes a list of transformers, where each transformer specifies:
1. A name (for reference).
2. The transformation to apply.
3. Which columns to apply it to.

Let's build separate transformations for numeric and categorical features:


In [ ]:
# Create preprocessing pipelines for each type

numeric_features = ['age', 'income']
nominal_features = ['city']         # No natural order
ordinal_features = ['education']    # Bachelor < Master < PhD

# Numeric: scale
numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

# Nominal: one-hot encode (no order exists between cities)
nominal_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Ordinal: encode preserving the meaningful order
ordinal_transformer = Pipeline([
    ('ordinal', OrdinalEncoder(
        categories=[['Bachelor', 'Master', 'PhD']]
    ))
])

print(f"Numeric features:  {numeric_features}")
print(f"Nominal features:  {nominal_features}")
print(f"Ordinal features:  {ordinal_features}")

Numeric features:  ['age', 'income']
Nominal features:  ['city']
Ordinal features:  ['education']


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('nom', nominal_transformer, nominal_features),
        ('ord', ordinal_transformer, ordinal_features)
    ])

print("ColumnTransformer created:")
print(preprocessor)

ColumnTransformer created:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['age', 'income']),
                                ('nom',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['city']),
                                ('ord',
                                 Pipeline(steps=[('ordinal',
                                                  OrdinalEncoder(categories=[['Bachelor',
                                                                              'Master',
                                                                              'PhD']]))]),
                                 ['education'])])


In [ ]:
# Fit and transform the data
X_transformed = preprocessor.fit_transform(df)

print("Original shape:", df.shape)
print("Transformed shape:", X_transformed.shape)
print("\nTransformed data (first 3 rows):")
print(X_transformed[:3])


Original shape: (5, 4)
Transformed shape: (5, 6)

Transformed data (first 3 rows):
[[-1.41421356 -1.36553779  0.          0.          1.          0.        ]
 [-0.70710678 -0.7152817   0.          1.          0.          1.        ]
 [ 0.          0.09753841  1.          0.          0.          0.        ]]


### What happened?

Original: 5 rows × 4 columns  
Transformed: 5 rows × **6 columns**

**Breakdown:**

- Numeric columns (2): `age`, `income` → 2 scaled columns  
- Nominal column – `city` (3 unique values: NYC, LA, Chicago) → 3 binary columns (one-hot encoded)  
- Ordinal column – `education` (3 ordered levels) → **1 numeric column** (e.g. Bachelor=0, Master=1, PhD=2)

Total: 2 + 3 + 1 = **6 columns**

**Why treat `education` differently from `city`?**  
Because education levels have a meaningful order (Bachelor < Master < PhD).  
`OrdinalEncoder` preserves this order, but it also introduces numeric spacing between categories.  
This means it should only be used when the ordering is meaningful and the model can interpret it appropriately.

In [ ]:
# Get feature names after transformation
# This shows exactly which column in the output corresponds to which original feature
numeric_names = numeric_features
nominal_names = preprocessor.named_transformers_['nom']['onehot'].get_feature_names_out(nominal_features).tolist()
ordinal_names = ordinal_features

feature_names = numeric_names + nominal_names + ordinal_names
print("Feature names after transformation:")
for i, name in enumerate(feature_names):
    print(f"  Column {i}: {name}")


Feature names after transformation:
  Column 0: age
  Column 1: income
  Column 2: city_Chicago
  Column 3: city_LA
  Column 4: city_NYC
  Column 5: education


>Note: In this introductory section, we apply fit_transform directly to the full dataset to keep the example simple. In any real workflow, you must split your data first – the Titanic example later in this notebook follows the correct approach.

## Handling missing values with SimpleImputer

Most real-world datasets contain missing values. Sklearn models cannot handle missing values by default and will raise an error, so we must deal with them before training.

The naive approach is to fill missing values using statistics calculated from the entire dataset – but this causes data leakage. If you calculate the mean using all rows before splitting, the test set has influenced the fill value.

`SimpleImputer` solves this by following the same fit/transform pattern as every other sklearn object – it learns the fill value from training data only, then applies it consistently to new data.

In [ ]:
# Create dataset with missing values
df_missing = pd.DataFrame({
    'age': [25, 30, None, 40, 45],
    'income': [30000, None, 75000, 85000, 120000],
    'city': ['NYC', 'LA', None, 'NYC', 'LA'],
    'education': ['Bachelor', 'Master', 'Bachelor', 'PhD', 'Master']
})

print("Dataset with missing values:")
print(df_missing)
print("\nMissing values per column:")
print(df_missing.isnull().sum())


Dataset with missing values:
    age    income  city education
0  25.0   30000.0   NYC  Bachelor
1  30.0       NaN    LA    Master
2   NaN   75000.0  None  Bachelor
3  40.0   85000.0   NYC       PhD
4  45.0  120000.0    LA    Master

Missing values per column:
age          1
income       1
city         1
education    0
dtype: int64


In [ ]:
# Numeric: impute with median, then scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Nominal: impute with most frequent, then one-hot encode
nominal_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Ordinal: no missing values in this example but good practice to include imputer
ordinal_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Bachelor', 'Master', 'PhD']]))
])

preprocessor_with_imputation = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('nom', nominal_transformer, nominal_features),
        ('ord', ordinal_transformer, ordinal_features)
    ])

print("Updated preprocessor with imputation:")
print(preprocessor_with_imputation)


Updated preprocessor with imputation:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'income']),
                                ('nom',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['city']),
                                ('ord',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ordinal'

In [ ]:
# Transform the data with missing values
X_transformed = preprocessor_with_imputation.fit_transform(df_missing)

print("Transformed data (with imputed values):")
print(X_transformed)
print("\nNo missing values in transformed data!")
print(f"All values finite: {np.all(np.isfinite(X_transformed))}")


Transformed data (with imputed values):
[[-1.41421356 -1.6701336   0.          1.          0.          0.        ]
 [-0.70710678  0.0695889   1.          0.          0.          1.        ]
 [ 0.         -0.10438335  0.          0.          1.          0.        ]
 [ 0.70710678  0.24356115  0.          1.          0.          2.        ]
 [ 1.41421356  1.4613669   1.          0.          0.          1.        ]]

No missing values in transformed data!
All values finite: True


## The remainder argument

What happens to columns you don't specify in the transformers? The `remainder` argument controls this.


In [ ]:
# Add an extra column that we won't transform
df_extra = df.copy()
df_extra['id'] = [1, 2, 3, 4, 5]

print("Dataset with extra 'id' column:")
print(df_extra)


Dataset with extra 'id' column:
   age  income     city education  id
0   25   30000      NYC  Bachelor   1
1   30   50000       LA    Master   2
2   35   75000  Chicago  Bachelor   3
3   40   85000      NYC       PhD   4
4   45  120000       LA    Master   5


In [ ]:
# Option 1: remainder='drop' (default)
# Drops any columns not explicitly transformed
preprocessor_drop = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('nom', OneHotEncoder(), nominal_features)
    ],
    remainder='drop'  # any column not listed above will be dropped
)

X_drop = preprocessor_drop.fit_transform(df_extra)
print("With remainder='drop':")
print(f"  Original columns: {df_extra.shape[1]}")
print(f"  Transformed columns: {X_drop.shape[1]}  (2 scaled + 3 one-hot)")
print("  Dropped columns: 'education' and 'id' — neither was listed in transformers")

With remainder='drop':
  Original columns: 5
  Transformed columns: 5  (2 scaled + 3 one-hot)
  Dropped columns: 'education' and 'id' — neither was listed in transformers


In [ ]:
preprocessor_pass = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('nom', OneHotEncoder(), nominal_features)
    ],
    remainder='passthrough'  # any unlisted column passes through unchanged
)

X_pass = preprocessor_pass.fit_transform(df_extra)
print("With remainder='passthrough':")
print(f"  Original columns: {df_extra.shape[1]}")
print(f"  Transformed columns: {X_pass.shape[1]}  (2 scaled + 3 one-hot + 2 passthrough)")
print("  Passed through unchanged: 'education' and 'id' — neither was listed in transformers")
print(f"\nLast two columns (education, id):")
print(f"  education: {X_pass[:, -2]}")
print(f"  id:        {X_pass[:, -1]}")

With remainder='passthrough':
  Original columns: 5
  Transformed columns: 7  (2 scaled + 3 one-hot + 2 passthrough)
  Passed through unchanged: 'education' and 'id' — neither was listed in transformers

Last two columns (education, id):
  education: ['Bachelor' 'Master' 'Bachelor' 'PhD' 'Master']
  id:        [1 2 3 4 5]


**When to use each:**

- **`remainder='drop'`** (default): Use when you only want the specified columns in your model. This is the safest choice – it makes your intentions explicit.

- **`remainder='passthrough'`**: Use when you have columns that don't need transformation but should still be included (e.g. already-scaled features, IDs for later reference).

**Best practice:** Always be explicit about which columns you're transforming. Don't rely on remainder to handle important features.


## Complete real-world example: Titanic dataset

Let's put it all together with a real dataset that has:
- missing values,
- mixed data types (numeric and categorical),
- a classification task.

We'll build a complete end-to-end pipeline from raw data to predictions.


In [1]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Load Titanic dataset
titanic = sns.load_dataset('titanic')

# Select relevant columns
df = titanic[['age', 'fare', 'sex', 'embarked', 'survived']].copy()

print("Titanic dataset:")
print(df.head(10))
print(f"\nShape: {df.shape}")
print("\nMissing values:")
print(df.isnull().sum())


Titanic dataset:
    age     fare     sex embarked  survived
0  22.0   7.2500    male        S         0
1  38.0  71.2833  female        C         1
2  26.0   7.9250  female        S         1
3  35.0  53.1000  female        S         1
4  35.0   8.0500    male        S         0
5   NaN   8.4583    male        Q         0
6  54.0  51.8625    male        S         0
7   2.0  21.0750    male        S         0
8  27.0  11.1333  female        S         1
9  14.0  30.0708  female        C         1

Shape: (891, 5)

Missing values:
age         177
fare          0
sex           0
embarked      2
survived      0
dtype: int64


In [2]:
# Separate features and target
X = df.drop('survived', axis=1)
y = df['survived']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


Training set: (712, 4)
Test set: (179, 4)


In [3]:
# Define feature groups
numeric_features = ['age', 'fare']
categorical_features = ['sex', 'embarked']

print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")


Numeric features: ['age', 'fare']
Categorical features: ['sex', 'embarked']


In [6]:
# Create preprocessing pipelines

# Numeric: impute missing values, then scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical: impute missing values, then one-hot encode
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessor created:")
print(preprocessor)


Preprocessor created:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'fare']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['sex', 'embarked'])])


In [7]:
# Create full pipeline: preprocessing + model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

print("Complete pipeline:")
print(full_pipeline)


Complete pipeline:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                               

In [8]:
# Train the pipeline
print("Training the model...")
full_pipeline.fit(X_train, y_train)

# Evaluate
train_accuracy = full_pipeline.score(X_train, y_train)
test_accuracy = full_pipeline.score(X_test, y_test)

print("\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"Training accuracy: {train_accuracy:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")


Training the model...

RESULTS
Training accuracy: 0.784
Test accuracy: 0.777


In [9]:
# Make predictions
y_pred = full_pipeline.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))



Classification Report:
              precision    recall  f1-score   support

        Died       0.80      0.85      0.82       110
    Survived       0.73      0.67      0.70        69

    accuracy                           0.78       179
   macro avg       0.77      0.76      0.76       179
weighted avg       0.77      0.78      0.77       179



## Inspecting the preprocessor

Let's look inside the preprocessor to see what it learned:


In [10]:
# Access the fitted preprocessor
fitted_preprocessor = full_pipeline.named_steps['preprocessor']

# Access numeric transformer
numeric_pipe = fitted_preprocessor.named_transformers_['num']
imputer = numeric_pipe.named_steps['imputer']
scaler = numeric_pipe.named_steps['scaler']

print("Numeric Preprocessing:")
print(f"  Missing values filled with: {imputer.statistics_}")
print(f"  Features scaled to mean=0, std=1")
print(f"  Original means: {scaler.mean_}")
print(f"  Original stds: {scaler.scale_}")

# Access categorical transformer
cat_pipe = fitted_preprocessor.named_transformers_['cat']
cat_imputer = cat_pipe.named_steps['imputer']
encoder = cat_pipe.named_steps['onehot']

print("\nCategorical preprocessing:")
print(f"  Missing values filled with: {cat_imputer.statistics_}")
print(f"  Encoded categories:")
for i, feature in enumerate(categorical_features):
    print(f"    {feature}: {encoder.categories_[i]}")


Numeric Preprocessing:
  Missing values filled with: [28.5    14.4542]
  Features scaled to mean=0, std=1
  Original means: [29.55606742 31.81982626]
  Original stds: [13.01612245 48.02534305]

Categorical preprocessing:
  Missing values filled with: ['male' 'S']
  Encoded categories:
    sex: ['female' 'male']
    embarked: ['C' 'Q' 'S']


## Saving the complete pipeline

Just like before, we can save the entire pipeline (preprocessing + model) for later use:


In [11]:
import joblib
from datetime import datetime

# Create descriptive filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
pipeline_filename = f'titanic_pipeline_{timestamp}.pkl'

# Save pipeline
joblib.dump(full_pipeline, pipeline_filename)
print(f"Pipeline saved to: {pipeline_filename}")

# Save metadata
metadata = {
    'model': 'Titanic Survival Classifier',
    'test_accuracy': test_accuracy,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'n_samples_train': len(X_train),
    'n_samples_test': len(X_test),
    'timestamp': timestamp
}

metadata_filename = f'titanic_pipeline_{timestamp}_metadata.pkl'
joblib.dump(metadata, metadata_filename)
print(f"Metadata saved to: {metadata_filename}")


Pipeline saved to: titanic_pipeline_20260430_080345.pkl
Metadata saved to: titanic_pipeline_20260430_080345_metadata.pkl


In [ ]:
# Load and use the saved pipeline
loaded_pipeline = joblib.load(pipeline_filename)
print("Pipeline loaded successfully!")

# Make a prediction on new data
new_passenger = pd.DataFrame({
    'age': [22],
    'fare': [7.25],
    'sex': ['male'],
    'embarked': ['S']
})

prediction = loaded_pipeline.predict(new_passenger)
probability = loaded_pipeline.predict_proba(new_passenger)

print(f"\nNew passenger prediction:")
print(f"  Input: {new_passenger.values[0]}")
print(f"  Predicted: {'Survived' if prediction[0] == 1 else 'Died'}")
print(f"  Confidence: {probability[0][prediction[0]]:.2%}")


Pipeline loaded successfully!

New passenger prediction:
  Input: [22 7.25 'male' 'S']
  Predicted: Died
  Confidence: 100.00%


## Key takeaways

**ColumnTransformer is essential for real-world data:**
- Apply different preprocessing to different column types.
- Handles numeric, nominal and ordinal features in one object.
- Prevents common mistakes (like scaling categorical data).



## Testing pipeline robustness

Before deploying any pipeline, verify it handles real-world edge cases gracefully. Two situations come up constantly in production: unseen categories and missing values in new data.


In [ ]:
# Test 1: Unseen category – 'T' was never in the training data
new_passenger_unseen = pd.DataFrame({
    'age': [28],
    'fare': [15.0],
    'sex': ['male'],
    'embarked': ['T']   # 'T' never appeared in training data
})

pred = full_pipeline.predict(new_passenger_unseen)
print("Test 1 — unseen embarked value 'T':")
print(f"  Prediction: {'Survived' if pred[0] == 1 else 'Died'}")
print("  Pipeline handled it without error.")
print("  Reason: handle_unknown='ignore' encodes unseen categories as all zeros.")

# Test 2: Missing values in new data
new_passenger_missing = pd.DataFrame({
    'age': [None],      # Missing age
    'fare': [22.0],
    'sex': [None],      # Missing sex
    'embarked': ['S']
})

pred = full_pipeline.predict(new_passenger_missing)
print("\nTest 2 — missing age and sex:")
print(f"  Prediction: {'Survived' if pred[0] == 1 else 'Died'}")
print("  Pipeline handled it without error.")
print("  Reason: SimpleImputer fills with the median/most_frequent learned from training data.")

# Test 3: Explain what would break without these safeguards
print("\nTest 3 – what would break without these safeguards:")
print("  Without handle_unknown='ignore':")
print("    → ValueError: Found unknown categories during transform")
print("  Without SimpleImputer inside the pipeline:")
print("    → ValueError: Input contains NaN")
print("\nBoth are essential for production-ready pipelines.")


Test 1 — unseen embarked value 'T':
  Prediction: Died
  Pipeline handled it without error.
  Reason: handle_unknown='ignore' encodes unseen categories as all zeros.

Test 2 — missing age and sex:
  Prediction: Died
  Pipeline handled it without error.
  Reason: SimpleImputer fills with the median/most_frequent learned from training data.

Test 3 – what would break without these safeguards:
  Without handle_unknown='ignore':
    → ValueError: Found unknown categories during transform
  Without SimpleImputer inside the pipeline:
    → ValueError: Input contains NaN

Both are essential for production-ready pipelines.


In [ ]:
# Cleanup (optional)
import os

files_to_remove = [pipeline_filename, metadata_filename]

for file in files_to_remove:
    if os.path.exists(file):
        os.remove(file)
        print(f"✓ Removed: {file}")


✓ Removed: titanic_pipeline_20260430_075444.pkl
✓ Removed: titanic_pipeline_20260430_075444_metadata.pkl
